K-1. Menyiapkan Lingkungan dan Data Contoh

In [44]:
!pip install q duckdb networkx shapely tinydb "deltalake>1.0"

from google.colab import drive
drive.mount("/content/drive")

DIR_P3 = "/content/drive/MyDrive/Colab Notebook/Big Data/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/Colab Notebook/Big Data/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"

import os
# hasil Praktikum 3 (K-10)
os.makedirs(DIR_SIMPAN, exist_ok=True)
assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Jalankan ulang P3/K-10 atau pakai K-1b."

import os, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd
import pyarrow.dataset as ds

# Pengukur waktu (dari Praktikum 1/K-3, versi ringkas)
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

# Membaca data dengan aman menggunakan pyarrow dataset untuk menghindari error unify dictionaries
def baca_parquet_dataset(path):
    dataset = ds.dataset(path, format="parquet", partitioning="hive")
    return dataset.to_table().to_pandas(categories=[])

df = ukur("baca trips_bersih", lambda: baca_parquet_dataset(TRIPS_BERSIH))

# Ganti nama kolom asli data TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

# Contoh 300.000 baris untuk K-2 dan K-3 (agar tidak terlalu lama)
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[baca trips_bersih] 3.8403 s
baris: 2,986,910 | kolom: 18


K-2. Tabel Biasa vs Key-value

In [45]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 key-value: kamus (dict) dengan kunci "perjalanan:<id>", nilainya teks JSON
def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}
kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)") # buat indeks
r2 = ukur("cari 200 perjalanan SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks) # setiap nilai harus dibuka dulu
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 1.5163 s
[key-value: bangun dict] 5.6539 s
jumlah key: 300000
[cari 200 perjalanan SQL tanpa indeks] 10.9808 s
[cari 200 perjalanan SQL dengan indeks] 0.0043 s
[cari 200 perjalanan key-value] 0.0016 s
hasil sama? True
[hitung per wilayah SQL GROUP BY] 0.2147 s
[hitung per wilayah key-value (buka semua nilai)] 1.3453 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3787, 39.13)]
hasil sama? True


K-3. Document: Bentuk Data yang Fleksibel

In [46]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {
            "jarak_mil": float(r.jarak_mil),
            "total": float(r.total_bayar),
            "metode_bayar": r.metode_bayar
        },
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

Q = Query()
print("jemput di Manhattan & total > 50: ", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip    : ", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip    : ", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen tanpa mengubah struktur apa pun
tabel.insert({
    "id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
    "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
    "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
    "cuaca": {"hujan": True, "suhu_c": -2.0}
})

print("dokumen yang punya field cuaca  : ", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang          : ", len(tabel))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu kredit",
    "tip": 3.22
  }
}
jemput di Manhattan & total > 50:  693
dokumen yang punya field tip    :  15906
dokumen yang TIDAK punya tip    :  4094
dokumen yang punya field cuaca  :  1
total dokumen sekarang          :  20001


K-4. Penyimpanan per Kolom: DuckDB pada Parquet

In [47]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
con_d.sql(f"""
CREATE VIEW perjalanan AS
SELECT tpep_pickup_datetime AS waktu_jemput,
       PULocationID AS zona_jemput,
       DOLocationID AS zona_tujuan,
       borough_naik AS wilayah_jemput,
       trip_distance AS jarak_mil,
       total_amount AS total_bayar,
       nama_pembayaran AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)
""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""

hasil_d = ukur("DuckDB hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))

hasil_s = ukur("SQLite hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1"""
).fetchall())

print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta (partition pruning, P3)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB hitung per wilayah (langsung dari Parquet)] 0.4293 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  2659609       22.72
1                      Queens   270315       70.46
2                     Unknown    37609       39.24
3                    Brooklyn    15724       32.72
4                       Bronx     3074       34.44
5  __HIVE_DEFAULT_PARTITION__      340       74.72
6               Staten Island      211       74.07
7                         EWR       28      106.41
[SQLite muat seluruh baris ke tabel] 16.4675 s
[SQLite hitung per wilayah] 1.8644 s
hasil sama? False
[DuckDB hanya wilayah Staten Island] 0.0563 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


K-5. Graph: Pertanyaan tentang Keterhubungan

In [48]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2
""").df())

# Zona = node (titik), arus = edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))

print("node (zona):", G.number_of_nodes(), " edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])

top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

MIN = 50
zona_awal = populer[0][0] # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?
# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN: continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}
    """).df()["tujuan"])

hg = ukur("2 langkah graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, " terjangkau (graph):", len(hg), "| (SQL):", len(hs), " sama?", hg == hs)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[hitung arus antar zona] 0.4207 s
node (zona): 262  edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL  : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]
[2 langkah graph (NetworkX)] 0.0096 s
[2 langkah SQL self-join (DuckDB)] 0.0082 s
zona awal: 236  terjangkau (graph): 214 | (SQL): 214  sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


K-6. Indeks Spasial

In [49]:
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
lintang = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70) # longitude
bujur = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)      # latitude
titik = points(bujur, lintang)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 STRtree, 1000 kotak",
             lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 loop Python, 5 kotak] 12.7024 s
[cara 2 NumPy, 1000 kotak] 0.5375 s
[bangun indeks STRtree] 0.0665 s
[cara 3 STRtree, 1000 kotak] 0.0064 s
5 kotak pertama loop: [0, 0, 0, 0, 0] | STRtree: [0, 0, 0, 0, 0]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 0


K-7. Delta Lake: Versi, Time Travel, dan Schema Enforcement

In [50]:
from deltalake import DeltaTable, write_deltalake
import shutil
import os

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

# Menggunakan penyimpanan lokal Colab untuk menghindari kendala filesystem Google Drive dengan Delta Lake
PATH_DELTA_LOKAL = "/content/trips_delta"
if os.path.exists(PATH_DELTA_LOKAL):
    shutil.rmtree(PATH_DELTA_LOKAL)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA_LOKAL, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA_LOKAL)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA_LOKAL, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA_LOKAL)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data. Anggap tarif Tunai di wilayah Bronx perlu ditambah 1.0
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)

dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA_LOKAL).version())

# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA_LOKAL, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA_LOKAL, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA_LOKAL).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), " terbaru:", len(terbaru))

selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), " jumlah baris dikoreksi:", n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA_LOKAL).history():
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA_LOKAL}/_delta_log")))

# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK, tabel tidak berubah
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA_LOKAL, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)

print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA_LOKAL).version())

# Opsional: Jika Anda ingin mem-backup hasilnya ke Google Drive Anda
# PATH_DELTA_DRIVE = f"{DIR_SIMPAN}/trips_delta"
# if os.path.exists(PATH_DELTA_DRIVE):
#     shutil.rmtree(PATH_DELTA_DRIVE)
# shutil.copytree(PATH_DELTA_LOKAL, PATH_DELTA_DRIVE)
# print("Delta table berhasil dicadangkan ke Google Drive!")

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 20
versi: 2
baris v0: 100000 | v1: 150000  terbaru: 150000
selisih total_bayar (terbaru - v1): 20.0  jumlah baris dikoreksi: 20
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


K-8. Small File Problem pada Tabel Delta

In [51]:
dt = DeltaTable(PATH_DELTA_LOKAL)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris) seperti data yang masuk sedikit demi sedikit
for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500 : 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA_LOKAL, potongan, mode="append")

dt = DeltaTable(PATH_DELTA_LOKAL)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); DeltaTable(PATH_DELTA_LOKAL).to_pandas(); t_sebelum = time.perf_counter() - t0

# COMPACTION: gabungkan file kecil menjadi file yang lebih besar (dicatat sebagai versi baru)
hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA_LOKAL)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA_LOKAL).to_pandas(); t_sesudah = time.perf_counter() - t0

print("file data aktif setelah compact:", len(dt.file_uris()), " | versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yang menghapusnya.
print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA_LOKAL}/*.parquet")))

kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True) # dry_run hanya melaporkan
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1  | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0828 s | sesudah: 0.0550 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


K-9. Merangkum dan Memilih

In [52]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                          langkah   detik
                                baca trips_bersih  3.8403
                           relasional: muat tabel  1.5163
                           key-value: bangun dict  5.6539
             cari 200 perjalanan SQL tanpa indeks 10.9808
            cari 200 perjalanan SQL dengan indeks  0.0043
                    cari 200 perjalanan key-value  0.0016
                  hitung per wilayah SQL GROUP BY  0.2147
  hitung per wilayah key-value (buka semua nilai)  1.3453
DuckDB hitung per wilayah (langsung dari Parquet)  0.4293
               SQLite muat seluruh baris ke tabel 16.4675
                        SQLite hitung per wilayah  1.8644
               DuckDB hanya wilayah Staten Island  0.0563
                           hitung arus antar zona  0.4207
                       2 langkah graph (NetworkX)  0.0096
                 2 langkah SQL self-join (DuckDB)  0.0082
                      cara 1 loop Python, 5 kotak 12.7024
              

Latihan 1: Pencarian 2.000 ID (Indeks SQL vs Key-Value)

In [53]:
# Menyiapkan 2.000 ID acak
random.seed(42)
id_uji_2000 = random.sample(range(1, len(data) + 1), 2000)

def cari_sql_2000():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji_2000]

def cari_kv_2000():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji_2000]

# Pastikan indeks sudah dibuat dari sel K-2 sebelumnya
waktu_sql = ukur("SQL (indeks) 2000 pencarian", cari_sql_2000)
waktu_kv = ukur("Key-value 2000 pencarian", cari_kv_2000)

# Menghitung rata-rata dalam mikrodetik
waktu_total_sql = catatan[-2]["detik"]
waktu_total_kv = catatan[-1]["detik"]

print(f"Rata-rata SQL (indeks): {(waktu_total_sql / 2000) * 1_000_000:.2f} mikrodetik/pencarian")
print(f"Rata-rata Key-value   : {(waktu_total_kv / 2000) * 1_000_000:.2f} mikrodetik/pencarian")

[SQL (indeks) 2000 pencarian] 0.0659 s
[Key-value 2000 pencarian] 0.0143 s
Rata-rata SQL (indeks): 32.95 mikrodetik/pencarian
Rata-rata Key-value   : 7.15 mikrodetik/pencarian


Latihan 2: Perbandingan Pencarian Dokumen (TinyDB) dan Tabel (SQLite)

In [54]:
# Misal kita mencari zona tujuan 164 dengan metode bayar Tunai
zona_target = 164

# Cara TinyDB (K-3)
Q = Query()
hasil_tinydb = tabel.search((Q.tujuan.zona == zona_target) & (Q.biaya.metode_bayar == "Tunai"))
print(f"Jumlah TinyDB: {len(hasil_tinydb)}")

# Cara SQL SQLite (K-2)
hasil_sql = con.execute("SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = 'Tunai'", (zona_target,)).fetchone()[0]
print(f"Jumlah SQL   : {hasil_sql}")

Jumlah TinyDB: 93
Jumlah SQL   : 1086


Latihan 3: Column Pruning pada DuckDB

In [55]:
sql_1_kolom = "SELECT SUM(total_bayar) FROM perjalanan"
sql_3_kolom = "SELECT SUM(total_bayar), SUM(jarak_mil), AVG(zona_jemput) FROM perjalanan"

ukur("DuckDB SUM 1 kolom", lambda: con_d.sql(sql_1_kolom).df())
ukur("DuckDB SUM 3 kolom", lambda: con_d.sql(sql_3_kolom).df())

[DuckDB SUM 1 kolom] 0.2749 s
[DuckDB SUM 3 kolom] 0.5556 s


,sum(total_bayar),sum(jarak_mil),avg(zona_jemput)
0,8.162144e+07,10231889.61,166.544273


Latihan 4: Menghitung Zona Tujuan Berbeda (Out-degree Graph vs SQL)

In [56]:
# 1. Cara Graph: Hitung tetangga terhubung dari setiap node (zona tujuan berbeda = jumlah successors)
out_degrees = [(n, len(list(G.successors(n)))) for n in G.nodes()]
top_5_graph = sorted(out_degrees, key=lambda x: (-x[1], x[0]))[:5]
print("Top 5 Graph (Out-degree):", top_5_graph)

# 2. Cara SQL: Hitung COUNT(DISTINCT) tujuan dari tabel arus
top_5_sql = con_d.sql("""
    SELECT asal, COUNT(DISTINCT tujuan) AS jml_tujuan
    FROM arus
    GROUP BY asal
    ORDER BY jml_tujuan DESC, asal ASC
    LIMIT 5
""").fetchall()
print("Top 5 SQL               :", top_5_sql)

Top 5 Graph (Out-degree): [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
Top 5 SQL               : [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]


Latihan 5: Pengaruh Ukuran Kotak pada Indeks Spasial

In [57]:
# Menguji dengan sisi kotak 0,02 derajat dan 0,001 derajat
for ukuran in [0.02, 0.001]:
    kotak_baru = [box(x - ukuran, y - ukuran, x + ukuran, y + ukuran) for x, y in zip(cx, cy)]

    def numpy_baru():
        hasil = []
        for x, y in zip(cx, cy):
            m = (bujur >= x - ukuran) & (bujur <= x + ukuran) & (lintang >= y - ukuran) & (lintang <= y + ukuran)
            hasil.append(int(m.sum()))
        return hasil

    ukur(f"NumPy kotak {ukuran}", numpy_baru)
    ukur(f"STRtree kotak {ukuran}", lambda: [len(pohon.query(k, predicate="contains")) for k in kotak_baru])

[NumPy kotak 0.02] 0.6421 s
[STRtree kotak 0.02] 0.0120 s
[NumPy kotak 0.001] 0.6977 s
[STRtree kotak 0.001] 0.0115 s


Latihan 6: Time Travel vs Restore di Delta Lake

In [58]:
# Pastikan PATH_DELTA sudah mengarah ke lokal atau '/content/trips_delta'
# seperti pada K-7 dan K-8 Anda
dt = DeltaTable(PATH_DELTA_LOKAL)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))

# Kembalikan tabel ke keadaan versi 1
dt.restore(1)

dt = DeltaTable(PATH_DELTA_LOKAL)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


tugas praktikum

In [59]:
data_tugas = sample[["id_perjalanan", "total_bayar", "metode_bayar", "tip"]].head(1000).copy()

# 1. MODEL TABEL (SQLite)
con_tugas = sqlite3.connect(":memory:") # Menggunakan memori sementara
data_tugas.to_sql("pembayaran", con_tugas, index=False)

mudah_sql = con_tugas.execute("SELECT metode_bayar, AVG(tip) FROM pembayaran GROUP BY metode_bayar").fetchall()
print("SQL Mudah (Agregasi):", mudah_sql)

# 2. MODEL DOCUMENT (TinyDB)
from tinydb import TinyDB
db_tugas = TinyDB(storage=MemoryStorage)
tabel_doc = db_tugas.table("pembayaran")
dokumen_bayar = data_tugas.to_dict(orient="records")
tabel_doc.insert_multiple(dokumen_bayar)

mudah_doc = len(tabel_doc.search(Query().metode_bayar == "Kartu kredit"))
print("Document Mudah (Pencarian fleksibel):", mudah_doc)

# 3. MODEL KEY-VALUE (Kamus Python)
kv_tugas = {f"bayar:{r['id_perjalanan']}": r for r in dokumen_bayar}

mudah_kv = kv_tugas["bayar:10"]
print("Key-Value Mudah (Ambil via ID):", mudah_kv)
canggung_kv = [v for k, v in kv_tugas.items() if v["metode_bayar"] == "Tunai"]

SQL Mudah (Agregasi): [(None, 3.632068965517241), ('Gratis', 0.0), ('Kartu kredit', 4.261797175866495), ('Sengketa', 0.0), ('Tunai', 0.0)]
Document Mudah (Pencarian fleksibel): 779
Key-Value Mudah (Ambil via ID): {'id_perjalanan': 10, 'total_bayar': 15.0, 'metode_bayar': 'Kartu kredit', 'tip': 1.0}


In [60]:
from deltalake import DeltaTable, write_deltalake
import os, shutil

PATH_TUGAS_DELTA = "/content/tugas_delta"
if os.path.exists(PATH_TUGAS_DELTA):
    shutil.rmtree(PATH_TUGAS_DELTA)

# --- HARI 1: Menulis sebagian data awal ---
data_hari_1 = df[KOLOM_DELTA].iloc[:5000].copy()
write_deltalake(PATH_TUGAS_DELTA, data_hari_1, mode="overwrite")
print("Hari 1 selesai. Versi tabel:", DeltaTable(PATH_TUGAS_DELTA).version())

# --- HARI 2: Menambah data dan mengoreksi baris ---
# 1. Menambah data baru
data_hari_2 = df[KOLOM_DELTA].iloc[5000:8000].copy()
write_deltalake(PATH_TUGAS_DELTA, data_hari_2, mode="append")

# 2. Mengoreksi data (Misal: diskon 1 dolar untuk jarak_mil < 1)
dt_tugas = DeltaTable(PATH_TUGAS_DELTA)
dt_tugas.update(predicate="jarak_mil < 1.0", updates={"total_bayar": "total_bayar - 1.0"})
print("Hari 2 selesai. Versi tabel aktif:", DeltaTable(PATH_TUGAS_DELTA).version())

# --- BUKTI TIME TRAVEL: Membaca keadaan akhir Hari 1 (Versi 0) ---
tabel_hari_1 = DeltaTable(PATH_TUGAS_DELTA, version=0).to_pandas()
print(f"Jumlah baris saat time travel ke Hari 1: {len(tabel_hari_1)} baris (seharusnya 5000)")

# --- LAMPIRAN ISI _delta_log ---
print("Isi riwayat _delta_log/ :")
print(sorted(os.listdir(f"{PATH_TUGAS_DELTA}/_delta_log")))

Hari 1 selesai. Versi tabel: 0
Hari 2 selesai. Versi tabel aktif: 2
Jumlah baris saat time travel ke Hari 1: 5000 baris (seharusnya 5000)
Isi riwayat _delta_log/ :
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
